In [ ]:
## Load packages, env parameters
import os
import pandas as pd
import numpy as np
import scanpy as sc
import gc
from scipy import sparse
import anndata
import pybedtools
#pybedtools.helpers.set_bedtools_path("/jp_envs/envs/seeksoulmethyl/bin/")
from ALLCools.mcds import MCDS
from ALLCools.clustering import binarize_matrix, filter_regions, lsi, tsne, significant_pc_test, log_scale
from ALLCools.plot import *
from pathlib import Path
from harmonypy import run_harmony
from collections import defaultdict

In [ ]:
## Load methylation data in .mcds format
sample_meta_path = "../input_metadata/sample_data.txt"
black_list_path = "../input_metadata/hg38-blacklist.v2.bed"

meta = pd.read_csv(sample_meta_path, sep="\t")
required_cols = {"sample_id", "allcool_gen_matrix","filtered_barcode"}

missing = required_cols - set(meta.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

black_list_fraction = 0.2
exclude_chrom = ['chrM', 'chrY']

adatas = []
for _, row in meta.iterrows():
    sample_id = row["sample_id"]
    mcds_path = row["allcool_gen_matrix"]
    filtered_barcode_file = row["filtered_barcode"]
    readcount_df = pd.read_csv(filtered_barcode_file, header=0)
    readcount_df["barcode"] = (sample_id + "_" + readcount_df["barcode"]).astype(str).str.strip().str.upper()
    readcount_df.set_index("barcode",inplace=True)
    mcds = MCDS.open(str(mcds_path), var_dim="chrom20k")
    mcds = mcds.remove_black_list_region(
        black_list_path=black_list_path,
        f=black_list_fraction  # Features having overlap > f with any black list region will be removed.
        )
    mcds = mcds.remove_chromosome(exclude_chrom)
    adata = mcds.get_score_adata(mc_type="CGN", quant_type="hypo-score",sparse=False)
    adata.obs_names.name = "barcode"
    adata.obs_names = sample_id + "_" + adata.obs_names.astype(str).str.strip().str.upper()
    adata.obs["sample_id"] = sample_id
    adata.obs["sample_id_str"] = sample_id
    adata.obs = adata.obs.join(readcount_df[["reads_counts"]],how="left")
    
    valid_barcodes=readcount_df.index[readcount_df["reads_counts"] > 5000].astype(str).tolist()
    mask = adata.obs_names.isin(valid_barcodes)
    adata = adata[mask].copy()
    # binarize the hypo score matrix and transform to sparse matrix to save memory
    binarize_matrix(adata, cutoff=0.95)
    adata.X = sparse.csr_matrix(adata.X)
    
    if adata.n_obs > 0:
        adatas.append(adata)

    del mcds
    del readcount_df
    del valid_barcodes
    del mask
    del adata
    gc.collect()
combined = sc.concat(adatas, join="inner", label="sample_id", index_unique=None)
del adatas

## Binarize and LSI
filter_regions(combined)
lsi(combined, algorithm="arpack", obsm="X_lsi")
significant_pc_test(combined, p_cutoff=0.1, obsm="X_lsi", update=True)


In [ ]:
## Run harmony, umap and tsne
eps = 1e-6
X = combined.obsm['X_lsi']
adata_lsi = run_harmony(X + eps,
                 meta_data=combined.obs,
                 vars_use='sample_id_str',
                 random_state=0,
                 nclust=50,
                 max_iter_harmony=20)

combined.obsm['X_lsi_harmony'] = adata_lsi.Z_corr.T 

sc.pp.neighbors(combined, n_neighbors=25, use_rep='X_lsi_harmony') 
sc.tl.umap(combined)
sc.tl.tsne(combined,use_rep="X_lsi_harmony",random_state=0,perplexity=30,n_jobs=32)

sc.tl.leiden(combined, resolution=0.6, key_added="leiden_harmony",random_state=0)

In [ ]:
## Consensus clustering
n_runs = 20
resolution = 0.6

for seed in range(n_runs):
    sc.pp.neighbors(combined, use_rep="X_lsi_harmony", random_state=seed)
    sc.tl.leiden(
        combined,
        resolution=resolution,
        key_added=f"leiden_{seed}"
    )
    print("run " + str(seed) + " finished!")

leiden_df = combined.obs
leiden_cols = [c for c in leiden_df.columns if c.startswith("leiden_")]
assert (combined.obs_names == leiden_df.index).all(), "Cell order mismatch!"
combined.obs[leiden_cols] = leiden_df[leiden_cols]
ref = combined.obs["leiden_harmony"]

for i in range(0, n_runs):
    cur = combined.obs[f"leiden_{i}"]
    mapping = {}
    for c in cur.unique():
        overlap = ref[cur == c]
        mapping[c] = overlap.value_counts().idxmax()
    combined.obs[f"leiden2_{i}"] = cur.map(mapping)

leiden_cols = [f"leiden2_{i}" for i in range(n_runs)]
labels_df = combined.obs[leiden_cols]

def consensus_label(row):
    return row.value_counts().idxmax()

combined.obs["consensus_cluster"] = labels_df.apply(consensus_label, axis=1)

df = pd.DataFrame({
    "cell_id": combined.obs_names,
    "UMAP1": combined.obsm["X_umap"][:, 0],
    "UMAP2": combined.obsm["X_umap"][:, 1],
    "TSNE1": combined.obsm["X_tsne"][:, 0],
    "TSNE2": combined.obsm["X_tsne"][:, 1],
    "methy_consensus_cluster": combined.obs["consensus_cluster"].values
})

df.to_csv("tmp_dataset/methy_consensus_clustering.tsv", index=False, sep="\t")